# Profile metrics and anonymous case study
Section 4.4.3 and Appendix A.5 (Tables 6-10).

Default paths are relative: inputs under `data/`, results under
`outputs/08_profile_metrics/`. No data are included. Each `pred/` and
`norm/` directory supplies:
- `dataset/csv/papers.csv`: `id`, `publication_date`, `total_citations`.
- `dataset/csv/papers_collect_after_6months.csv`: `old_id`, `total_citations`.
- `dataset/csv/paper_authors.csv`: `paper_id`, `author_id`.
- `dataset/csv/citations.csv`: `citing_paper_id`, `cited_paper_id`.
- `metadata/edge_list.txt`: citing and cited paper keys, one directed edge per line.
- `metadata/louvain/best_louvain_reso_1.8.csv` (Predatory) or
  `best_louvain_reso_1.csv` (Normal): `Nodes`, comma-separated community members.

The community inputs are the saved Louvain seed-42 partitions, at resolutions
1.8 and 1.0 respectively. Author and paper keys remain internal. No author
registry is loaded; the case output contains only metric-ranked P/N labels.
Author-list equality is checked through the linked author-key sets; no raw byline
text is loaded.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

# Default paths are relative to the repository root.
REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '08_profile_metrics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = {"Predatory": "pred", "Normal": "norm"}
FIRST_COLLECTION = {"Predatory": pd.Timestamp("2024-09-09"),
                    "Normal": pd.Timestamp("2024-12-09")}
SECOND_COLLECTION = pd.Timestamp("2025-07-07")
OBSERVATION_DAYS = {label: (SECOND_COLLECTION-first).days
                    for label, first in FIRST_COLLECTION.items()}
DAYS_PER_YEAR = 365.25
DAYS_PER_MONTH = 30.4375

def save_table(frame, name):
    # Only aggregate statistics and anonymous case metrics are exported.
    frame.to_csv(OUTPUT_DIR/(name+".csv"), index=False)
    display(frame)

def monthly_change(value, label):
    return value*DAYS_PER_MONTH/OBSERVATION_DAYS[label]


from collections import defaultdict, Counter
from itertools import combinations
import networkx as nx
import statsmodels.formula.api as smf


In [ ]:
def match_papers(baseline, followup):
    for frame, key in [(baseline, "id"), (followup, "old_id")]:
        if frame[key].isna().any() or frame[key].duplicated().any():
            raise ValueError(f"Matching key {key!r} must be nonmissing and unique.")
    return baseline[["id", "publication_date", "total_citations"]].merge(
        followup[["old_id", "total_citations"]], left_on="id", right_on="old_id",
        suffixes=("_t1", "_t2"), validate="one_to_one"
    )

def finite_counts(frame):
    return np.isfinite(frame[["total_citations_t1", "total_citations_t2"]]).all(axis=1)


paper_data = {}
for label, group in DATASETS.items():
    folder = DATA_DIR/group/"dataset"/"csv"
    baseline = pd.read_csv(folder/"papers.csv", usecols=["id", "publication_date", "total_citations"])
    followup = pd.read_csv(folder/"papers_collect_after_6months.csv", usecols=["old_id", "total_citations"])
    for frame in (baseline, followup):
        frame["total_citations"] = pd.to_numeric(frame["total_citations"], errors="coerce")
    paper_data[label] = {"baseline": baseline, "followup": followup,
                         "matched": match_papers(baseline, followup)}


## Matched portfolios
Use the same finite, nonnegative integer-count papers in both snapshots for
each author's citation totals, h-index and i10-index. Deduplicate paper-author
links. Undated matched papers remain in metric calculations. The first year
is the earliest first four-digit year in all observed baseline linked papers,
including papers without a matched follow-up. No future-year truncation or
identity-based exclusion is imposed.

The recent cohort has first observed year 2022-2024. The single-year-history
subset additionally requires all dated baseline papers to fall in one year.
Annual and monthly changes use 365.25 and 30.4375 days respectively; zero and
negative changes are retained. Profile Cohen's d uses sample-size-pooled variance.

In [ ]:
def h_index(citation_counts):
    """citation_counts: iterable of per-paper citation counts for one author."""
    counts = sorted((c for c in citation_counts if pd.notna(c)), reverse=True)
    h = 0
    for i, c in enumerate(counts, start=1):
        if c >= i:
            h = i
        else:
            break
    return h

def i10_index(citation_counts):
    return int(sum(1 for c in citation_counts if pd.notna(c) and c >= 10))

def author_profile_metrics(paper_citation_map, paper_to_authors):
    """
    paper_citation_map: dict paper_id -> citation count (t1, t2, or adjusted)
    paper_to_authors:   dict paper_id -> set of author_id
    returns: DataFrame [author_id, n_papers, h_index, i10_index, total_citations]
    """
    author_to_counts = defaultdict(list)
    for paper_id, authors in paper_to_authors.items():
        c = paper_citation_map.get(paper_id)
        if c is None or pd.isna(c):
            continue
        for a in authors:
            author_to_counts[a].append(c)

    rows = []
    for author_id, counts in author_to_counts.items():
        rows.append({
            "author_id": author_id,
            "n_papers": len(counts),
            "h_index": h_index(counts),
            "i10_index": i10_index(counts),
            "total_citations": sum(counts),
        })
    return pd.DataFrame(rows)

BASE_METRICS = {"citation_growth": "total_citations", "h_index_growth": "h_index", "i10_index_growth": "i10_index"}
GROWTH_METRICS = {"citation_growth": "Citations", "h_index_growth": "h-index", "i10_index_growth": "i10-index"}
profiles, matched_papers = {}, {}
for label, data in paper_data.items():
    folder = DATA_DIR/DATASETS[label]/"dataset"/"csv"
    links = pd.read_csv(folder/"paper_authors.csv", usecols=["paper_id", "author_id"])
    if links.isna().any().any():
        raise ValueError("Paper-author keys must be nonmissing.")
    links = links.drop_duplicates(["paper_id", "author_id"])
    relations = links.groupby("paper_id").author_id.apply(set).to_dict()
    data["paper_to_authors"] = relations
    paired = data["matched"].loc[finite_counts(data["matched"])].set_index("id", drop=False)
    values = paired[["total_citations_t1", "total_citations_t2"]].to_numpy()
    assert (values >= 0).all() and np.equal(values, np.floor(values)).all()
    counts = {t: paired["total_citations_"+t].to_dict() for t in ["t1", "t2"]}
    matched_keys = links.loc[links.paper_id.isin(paired.index)].groupby("author_id").paper_id.apply(lambda x: tuple(sorted(set(x))))
    observed_keys = links.groupby("author_id").paper_id.apply(lambda x: tuple(sorted(set(x))))
    years = pd.to_numeric(data["baseline"].publication_date.astype(str).str.extract(r"(\d{4})", expand=False), errors="coerce")
    year_map = dict(zip(data["baseline"].id, years))
    rows = []
    for author, key in matched_keys.items():
        c1, c2 = [[counts[t][pid] for pid in key] for t in ["t1", "t2"]]
        dated = [int(year_map[pid]) for pid in observed_keys[author] if pd.notna(year_map.get(pid))]
        row = {"author_id": author, "dataset": label, "portfolio_key": key,
               "cluster_id": label+"|"+str(observed_keys[author]), "matched_paper_count": len(key),
               "first_year": min(dated) if dated else np.nan,
               "career_span": max(dated)-min(dated) if dated else np.nan,
               "total_citations_t1": sum(c1), "total_citations_t2": sum(c2),
               "h_index_t1": h_index(c1), "h_index_t2": h_index(c2),
               "i10_index_t1": i10_index(c1), "i10_index_t2": i10_index(c2)}
        for metric, base in BASE_METRICS.items():
            row[metric] = row[base+"_t2"]-row[base+"_t1"]
            row[metric+"_annual"] = row[metric]*DAYS_PER_YEAR/OBSERVATION_DAYS[label]
            row[metric+"_monthly"] = monthly_change(row[metric], label)
        rows.append(row)
    profiles[label] = pd.DataFrame(rows)
    matched_papers[label] = paired
    assert profiles[label].author_id.is_unique
    for t in ["t1", "t2"]:
        for metric in ["h_index", "i10_index"]:
            assert profiles[label][metric+"_"+t].between(0, profiles[label].matched_paper_count).all()

recent = {label: frame.loc[frame.first_year.isin([2022, 2023, 2024])].copy() for label, frame in profiles.items()}
single_year = {label: frame.loc[frame.first_year.isin([2022, 2023, 2024]) & frame.career_span.eq(0)].copy()
               for label, frame in profiles.items()}


In [ ]:
def profile_comparison(p, n):
    p, n = np.asarray(p, dtype=float), np.asarray(n, dtype=float)
    assert np.isfinite(p).all() and np.isfinite(n).all()
    pooled = np.sqrt(((len(p)-1)*p.var(ddof=1)+(len(n)-1)*n.var(ddof=1))/(len(p)+len(n)-2))
    t, tp = stats.ttest_ind(p, n, equal_var=False)
    u, up = stats.mannwhitneyu(p, n, alternative="two-sided")
    return {"Welch t": t, "Welch p": tp, "Mann-Whitney p": up,
            "Cohen d": (p.mean()-n.mean())/pooled if pooled else np.nan}

distribution = []
for title, column in [("h(t1)", "h_index_t1"), ("h(t2)", "h_index_t2"), ("Delta h", "h_index_growth"),
                      ("i10(t1)", "i10_index_t1"), ("i10(t2)", "i10_index_t2"), ("Delta i10", "i10_index_growth")]:
    p, n = [profiles[label][column] for label in DATASETS]
    comparison = profile_comparison(p, n)
    distribution.append({"Measure": title, "N Predatory": len(p), "N Normal": len(n),
                         "Predatory mean": p.mean(), "Predatory SD": p.std(ddof=1),
                         "Normal mean": n.mean(), "Normal SD": n.std(ddof=1),
                         "Cohen d": comparison["Cohen d"], "Mann-Whitney p": comparison["Mann-Whitney p"]})
save_table(pd.DataFrame(distribution), "table6_profile_distributions")

growth_rows = []
for population, frames in [("All matched", profiles), ("First observed 2022-2024", recent),
                            ("Single-year history 2022-2024", single_year)]:
    for label, frame in frames.items():
        for metric, title in GROWTH_METRICS.items():
            row = {"Population": population, "Dataset": label, "N": len(frame), "Metric": title}
            for suffix, units in [("", "Cumulative"), ("_annual", "Annual"), ("_monthly", "Monthly")]:
                row[units+" mean"] = frame[metric+suffix].mean()
                row[units+" median"] = frame[metric+suffix].median()
            growth_rows.append(row)
save_table(pd.DataFrame(growth_rows), "table7_profile_growth")


## Volume-adjusted regression (Table 8)
Fit OLS separately to each annualized gain in the recent cohort. The predictors
are set membership (Normal reference) and log(1 + matched paper count).
Cluster standard errors by the exact full observed baseline paper portfolio,
with dataset-local keys kept separate. This cluster definition includes unmatched
baseline papers; it differs from the matched portfolio used to compute outcomes.

In [ ]:
regression_data = pd.concat(recent.values(), ignore_index=True)
regression_data["log_volume"] = np.log1p(regression_data.matched_paper_count)
regression_rows = []
for metric, title in GROWTH_METRICS.items():
    model = smf.ols(metric+"_annual ~ C(dataset, Treatment(reference='Normal')) + log_volume",
                    data=regression_data).fit(cov_type="cluster", cov_kwds={"groups": regression_data.cluster_id})
    term = next(term for term in model.params.index if "dataset" in term)
    lower, upper = model.conf_int().loc[term]
    regression_rows.append({"Outcome": title, "N": int(model.nobs), "Clusters": regression_data.cluster_id.nunique(),
                            "Predatory coefficient": model.params[term], "95% CI lower": lower,
                            "95% CI upper": upper, "p-value": model.pvalues[term]})
save_table(pd.DataFrame(regression_rows), "table8_volume_adjusted_regression")


## Observed-edge removal (Table 9)
Self-citations have at least one author key shared by the citing and cited papers.
For repeated-target groups, use directed edges within each saved seed-42 Louvain
community. Retain citing papers with at least two observed targets. Connect two
citing papers if they share at least two targets and their target-set Jaccard
similarity is at least 0.80. Groups are connected components of size at least two;
the condition applies to graph edges, not every pair in a component. Count an
incoming edge only when at least two members of a group cite that target.

Remove each category separately from matched baseline counts, flooring at zero,
then recompute h and i10 on unchanged matched portfolios. For linked shares,
use the union of self-citation and repeated-target edges, counting overlap once.
Only observed edges can be removed; this is not a reconstruction of all citations.

In [ ]:
JACCARD_THRESHOLD = 0.80
MIN_SHARED_TARGETS = 2
def _build_similarity_graph(target_sets):
    """Verbatim logic from citation_network.ipynb Step 4."""
    graph = nx.Graph()
    graph.add_nodes_from(target_sets)
    shared_counts = Counter()
    target_to_citers = defaultdict(list)
    for citing_paper, targets in target_sets.items():
        for target in targets:
            target_to_citers[target].append(citing_paper)
    for citers in target_to_citers.values():
        for left, right in combinations(sorted(citers), 2):
            shared_counts[(left, right)] += 1
    for (left, right), shared in shared_counts.items():
        if shared < MIN_SHARED_TARGETS:
            continue
        union = len(target_sets[left] | target_sets[right])
        jaccard = shared / union
        if jaccard >= JACCARD_THRESHOLD:
            graph.add_edge(left, right, shared_targets=shared, jaccard=jaccard)
    return graph

def subtract_observed_edges(paper_citation_map, edge_counts):
    """Subtract observed incoming citation edges while preserving missing values.

    Returns the adjusted map and the number of papers for which the observed
    edge count exceeded the Google Scholar citation count and therefore
    required clipping at zero.
    """
    adjusted = {}
    clipped = 0
    for paper_id, count in paper_citation_map.items():
        if pd.isna(count):
            adjusted[paper_id] = np.nan
            continue
        removed = edge_counts.get(paper_id, 0)
        if removed > count:
            clipped += 1
        adjusted[paper_id] = max(0, count - removed)
    return adjusted, clipped

def self_citation_edges(citations, paper_to_authors):
    edges = set()
    for citing, cited in citations[["citing_paper_id", "cited_paper_id"]].drop_duplicates().itertuples(index=False, name=None):
        citing_authors, cited_authors = paper_to_authors.get(citing), paper_to_authors.get(cited)
        if citing_authors and cited_authors and citing_authors & cited_authors:
            edges.add((citing, cited))
    return edges

def repeated_target_counts_and_edges(communities, edges):
    """Preserve original removal counts and case-table union semantics in one pass."""
    counts, selected_edges = defaultdict(int), set()
    for nodes_text in communities["Nodes"]:
        nodes = {int(value.strip()) for value in str(nodes_text).split(",") if value.strip()}
        within = {(source, target) for source, target in edges if source in nodes and target in nodes}
        observed_targets = defaultdict(set)
        for source, target in within:
            observed_targets[source].add(target)
        eligible = {source: targets for source, targets in observed_targets.items() if len(targets) >= 2}
        if not eligible:
            continue
        graph = _build_similarity_graph(eligible)
        components = [sorted(component) for component in nx.connected_components(graph) if len(component) >= 2]
        for target in set().union(*eligible.values()):
            for members in components:
                citing_members = [paper for paper in members if target in eligible[paper]]
                if len(citing_members) >= 2:
                    counts[target] += len(citing_members)
                    selected_edges.update((paper, target) for paper in citing_members)
    return counts, selected_edges

supplementary_edges = {}
removal_profiles, removal_rows = {}, []
for label, group in DATASETS.items():
    folder = DATA_DIR/group
    citations = pd.read_csv(folder/"dataset"/"csv"/"citations.csv", usecols=["citing_paper_id", "cited_paper_id"])
    self_edges = self_citation_edges(citations, paper_data[label]["paper_to_authors"])
    resolution = "1.8" if label == "Predatory" else "1"
    communities = pd.read_csv(folder/"metadata"/"louvain"/("best_louvain_reso_"+resolution+".csv"), usecols=["Nodes"])
    with (folder/"metadata"/"edge_list.txt").open(encoding="utf-8") as stream:
        edges = {tuple(map(int, line.split()[:2])) for line in stream if line.strip()}
    assert all(len(edge) == 2 for edge in edges)
    group_counts, group_edges = repeated_target_counts_and_edges(communities, edges)
    assert dict(group_counts) == dict(Counter(target for _, target in group_edges))
    supplementary_edges[label] = {"self": self_edges, "group": group_edges}
    original = matched_papers[label].total_citations_t1.to_dict()
    for method, selected in [("Self-citation removed", self_edges), ("Repeated-target removed", group_edges)]:
        adjusted, clipped = subtract_observed_edges(original, Counter(target for _, target in selected))
        after = author_profile_metrics(adjusted, paper_data[label]["paper_to_authors"])
        before = profiles[label]
        assert set(after.author_id) == set(before.author_id)
        after = after.set_index("author_id").loc[before.author_id].reset_index()
        assert np.array_equal(after.n_papers, before.matched_paper_count)
        removal_profiles[(label, method)] = after
        for metric in ["h_index", "i10_index"]:
            first, last = before[metric+"_t1"].mean(), after[metric].mean()
            removal_rows.append({"Dataset": label, "Removal": method, "Metric": metric, "N": len(after),
                                 "Baseline mean": first, "After-removal mean": last,
                                 "Relative decrease (%)": 100*(first-last)/first if first else np.nan,
                                 "Papers clipped at zero": clipped})
save_table(pd.DataFrame(removal_rows), "table9_removal_summary")
removal_tests = []
for method in ["Self-citation removed", "Repeated-target removed"]:
    for metric in ["h_index", "i10_index"]:
        p, n = [removal_profiles[(label, method)][metric] for label in DATASETS]
        removal_tests.append({"Removal": method, "Metric": metric, **profile_comparison(p, n)})
save_table(pd.DataFrame(removal_tests), "table9_removal_tests")


## Top ten recently observed profiles
Within the first-observed-2022-2024 cohort, select ten per set by raw cumulative
citation gain, descending, breaking ties by the internal author key. Report the
mean annualized gain and the repeated-target share of their pooled baseline
citations. This share is citation-weighted rather than a mean of profile percentages.

In [ ]:
top_recent_rows = []
for label, frame in recent.items():
    selected = frame.sort_values(["citation_growth", "author_id"], ascending=[False, True], kind="stable").head(10)
    repeated_counts = Counter(target for _, target in supplementary_edges[label]["group"])
    repeated = sum(sum(repeated_counts.get(pid, 0) for pid in key) for key in selected.portfolio_key)
    baseline = selected.total_citations_t1.sum()
    top_recent_rows.append({"Dataset": label, "N": len(selected), "Annual mean citation gain": selected.citation_growth_annual.mean(),
                            "Repeated-target share (%)": 100*repeated/baseline if baseline else np.nan})
save_table(pd.DataFrame(top_recent_rows), "top_ten_recent_profiles")


## Anonymous 2024-cohort case study (Table 10)
The cohort is first observed in 2024; it is not restricted to single-year histories.
Rank independently within each set by matched baseline citation total, descending,
breaking ties by the internal author key. Labels P1-P10 and N1-N10 are assigned
only from these ranks. No identity registry or historical label crosswalk is used.
Linked shares use the union of observed edge categories over each matched portfolio.
For identical portfolios in the selected cases, check that the linked author
sets agree across all their papers. This checks membership, not the spelling or
order of a raw byline string.

In [ ]:
case_rows, overlap_rows = [], []
for label, frame in profiles.items():
    selected = frame.loc[frame.first_year.eq(2024)].sort_values(
        ["total_citations_t1", "author_id"], ascending=[False, True], kind="stable").head(10)
    prefix = "P" if label == "Predatory" else "N"
    linked_counts = Counter(target for _, target in supplementary_edges[label]["self"] | supplementary_edges[label]["group"])
    anonymous_portfolios = []
    for rank, (_, row) in enumerate(selected.iterrows(), 1):
        case = prefix+str(rank)
        key = row.portfolio_key
        linked = sum(linked_counts.get(pid, 0) for pid in key)
        case_rows.append({"Case": case, "Matched papers": row.matched_paper_count,
                          "Citations t1": row.total_citations_t1, "Citations t2": row.total_citations_t2,
                          "Citation gain": row.citation_growth,
                          "h t1": row.h_index_t1, "h t2": row.h_index_t2,
                          "i10 t1": row.i10_index_t1, "i10 t2": row.i10_index_t2,
                          "Linked citations": linked,
                          "Linked share (%)": 100*linked/row.total_citations_t1 if row.total_citations_t1 else np.nan})
        anonymous_portfolios.append((case, key, row.total_citations_t1))
    for left, right in combinations(anonymous_portfolios, 2):
        if left[1] == right[1]:
            author_sets = [frozenset(paper_data[label]["paper_to_authors"].get(pid, set())) for pid in left[1]]
            equal = bool(all(author_sets) and len(set(author_sets)) == 1)
            overlap_rows.append({"Case pair": left[0]+" / "+right[0], "Shared papers": len(left[1]),
                                 "Baseline citations": left[2], "Identical author sets": equal})
    del anonymous_portfolios
save_table(pd.DataFrame(case_rows), "table10_anonymous_cases")
save_table(pd.DataFrame(overlap_rows, columns=["Case pair", "Shared papers", "Baseline citations", "Identical author sets"]),
            "table10_identical_portfolios")
